# Motion Enhancer - train the AI flow model on Colab

Runtime -> Change runtime type -> **GPU** (T4 works; L4 or A100 are much faster). Then run the cells in order.

Put these three files in Google Drive, folder `MyDrive/MotionEnhancer/` (made on your PC, see `README.md`, section "Training on Google Colab"):

| File | What |
|---|---|
| `mefi.zip` | the trainer code (`tools/ai-train/mefi`) |
| `dataset.zip` | the dataset folder: `index.json` + `triplets/` |
| `init.pt` | the model to start from (e.g. `runs/last.pt`) |

Runs are saved to `MyDrive/MotionEnhancer/runs/`, so a disconnect loses nothing: run the **Continue** cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/MotionEnhancer'

In [ ]:
# Code and dataset onto the Colab machine's own disk (reading triplets from Drive is slow).
!rm -rf /content/work && mkdir -p /content/work/data
!unzip -q "$ROOT/mefi.zip" -d /content/work
!unzip -q "$ROOT/dataset.zip" -d /content/work/data
!ls /content/work /content/work/data | head
import json; print(len(json.load(open('/content/work/data/index.json'))), 'triplets')

## Settings

Batch by GPU memory (`large` model, crop 336-480): T4 16 GB -> 4-6, L4 24 GB -> 6-8, A100 40/80 GB -> 8-12.
The crop must fit inside the dataset's frames (352 px dataset: 336).
Start with these and lower the batch if it runs out of memory - on Colab that only stops the cell.

In [ ]:
RUN     = f'{ROOT}/runs/colab1'   # a NEW folder for a new run
INIT    = f'{ROOT}/init.pt'       # same model size as PRESET
PRESET  = 'large'
BATCH   = 4
CROP    = 336                     # must fit the frames: 336 for a 352 px dataset, 480 for 480 or 720
LR      = 1e-4                    # fine-tune; 1e-3 only from scratch (INIT = None)
EPOCHS  = 6
SUBPIXEL = True                   # small moves to full length (fine-tunes only)
WORKERS = 2

In [ ]:
import subprocess, json, sys, time

def train(extra):
    cmd = [sys.executable, '-u', '-m', 'mefi.train', '--data', '/content/work/data', '--run', RUN,
           '--workers', str(WORKERS)] + extra
    print(' '.join(cmd))
    p = subprocess.Popen(cmd, cwd='/content/work', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    last = 0.0
    for line in p.stdout:
        try:
            e = json.loads(line)
        except ValueError:
            print(line, end=''); continue
        kind = e.get('event')
        if kind == 'step':
            if time.time() - last >= 30:
                last = time.time()
                print(f"step {e['step']}/{e['steps']}  epoch {e['epoch']}  loss {e['loss']:.4f}  "
                      f"lr {e['lr']:.2e}  about {e['eta'] // 60} min left")
        elif kind == 'epoch':
            print(f"== epoch {e['epoch'] + 1}/{e['epochs']}: {e['psnr']:.2f} dB (plain blend {e['blend_psnr']:.2f}), "
                  f"best {e['best']:.2f}{'  <- new best.pt' if e.get('improved') else ''}")
        elif kind in ('done', 'warning', 'error', 'resumed', 'device', 'data', 'model', 'initialized'):
            print({k: v for k, v in e.items() if not isinstance(v, (dict, list))})
    return p.wait()

args = ['--preset', PRESET, '--refiner', '--pure-warp', '1', '--batch', str(BATCH), '--crop', str(CROP),
        '--lr', str(LR), '--epochs', str(EPOCHS)]
if INIT:
    args += ['--init-from', INIT]
if SUBPIXEL:
    args.append('--subpixel')
print('exit code', train(args))

## Continue after a disconnect

Re-run the first four cells (GPU check, Drive, unzip, settings), then this one. It picks up the same run with its saved settings.

In [ ]:
print('exit code', train(['--resume']))

## Done

Download `best.pt` from `MyDrive/MotionEnhancer/runs/colab1/` into `tools/ai-train/runs/colab1/` on your PC and export it there
with the trainer's Export page at the size the dataset was built for: 640x352 for a 352 px dataset, 832x480 for 480, 1280x720 for 720.